# CALE Behavior Matrix CFA

This notebook fits theory-guided CFA-style SEM models to CALE diagnostic behavior indicators. It is designed for server-side use after `experiment.py` has exported a behavior matrix CSV.

Interpretation note: `semopy` fits a continuous-indicator CFA approximation. CALE diagnostic indicators are bounded and often binary, so these results should be treated as preliminary internal-structure evidence rather than definitive ordinal CFA validation.

解释提醒：这个 notebook 不是为了证明 CALE 已经完成 psychometric validation，而是为了检验一个更温和也更稳妥的问题：CALE diagnostic indicators 是否表现出和理论一致的 multidimensional internal structure。

In [ ]:
# Uncomment on the server if semopy is not installed.
# %pip install semopy pandas numpy

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from semopy import Model, calc_stats

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## Core Questions / 核心问题

Run the notebook with these questions in mind:

1. **Is CALE behavior one-dimensional or multidimensional?**  
   CALE evaluator behavior 是不是只是一个 general quality factor，还是更适合被解释为多个相关但可区分的 latent dimensions？

2. **Does the theory-guided 3-factor model fit better than simpler baselines?**  
   如果 `three_factor` 比 `one_factor` / `two_factor` 明显更好，可以支持 multidimensional measurement story。

3. **Do indicators load onto the intended factors?**  
   比如 `source_faithfulness` 和 `evidence_grounding` 是否真的强烈反映 `factual_handling`？

4. **Are factors related but distinguishable?**  
   Factor correlations 如果中等偏高，说明 dimensions related but separable；如果接近 1，说明可能分不开。

5. **Is the structure robust across target models and CALE variants?**  
   Qwen-only / Llama-only / variant-specific 结果如果方向一致，就说明结构不是某个 target model 或 variant 的偶然产物。

6. **What can we claim?**  
   Strong claim: only if fit, loadings, factor correlations, and robustness all look good.  
   Safer claim: CFA-style results provide preliminary internal-structure evidence.

## What To Look At / 指标怎么看

| Output | What it means | Good sign | If good, we can say |
|---|---|---|---|
| CFI / TLI | Approximate comparative fit | Higher is better; around .90 acceptable, .95 strong | The hypothesized structure fits better than weak/null baseline |
| RMSEA | Approximate residual misfit per df | Lower is better; < .08 acceptable, < .06 strong | Model misfit is not too large |
| SRMR | Average standardized residual | Lower is better; < .08 usually acceptable | Reproduced correlations are close to observed correlations |
| AIC / BIC | Model comparison criteria | Lower is better among competing models | One model is more parsimonious / better supported than another |
| Standardized loadings | Indicator-factor relation | > .70 strong, .40-.70 moderate, < .30 weak | Indicators meaningfully reflect intended latent factor |
| Factor correlations | Relation between latent factors | .30-.80 often useful; > .90 may be too high | Factors are related but distinguishable |

中文速读：

- `three_factor` 明显优于 `one_factor`：可以说 CALE behavior is not merely a single score。
- 大多数 intended loadings 中高：可以说 indicators align with theorized dimensions。
- factor correlations 不接近 1：可以说 dimensions are related but empirically distinguishable。
- Qwen/Llama/variant robustness 相似：可以说 internal structure is not driven by a single target model or evaluator variant。
- 如果 fit 很差或 loading 很弱：不要强 claim；改说 current indicators need refinement 或 structure remains exploratory。

## Configure Paths

Set `DATA_PATH` to the behavior matrix CSV produced by `experiment.py --behavior-matrix-output`. The default below follows the current README convention for the full neutral FEVER run.

## CFA Input Choice / CFA 输入选择

The default CFA input is `heuristic_full`, because it is the only full-sample behavior matrix with all CALE variants.

默认 CFA 输入是 `heuristic_full`，因为它是唯一覆盖完整样本且包含所有 CALE variants 的 behavior matrix。

The Qwen2.5-7B and DeepSeek V4-Pro files can be used as exploratory backend-robustness CFA-style checks, but they are subset analyses and should not replace the main CFA.

Qwen2.5-7B 和 DeepSeek V4-Pro 文件可以作为探索性的 backend-robustness CFA-style checks，但它们是 subset analyses，不应该替代主 CFA。


In [ ]:
# Choose the behavior matrix you want to analyze.
# Default CFA input is the main heuristic full behavior matrix, because it has
# the full sample and all CALE variants. Strong-evaluator subsets are useful for
# backend robustness checks, but they are smaller and usually only include
# direct_llm_judge + full_attack_aware_cale.

DATA_PATHS = {
    "heuristic_full": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_behavior_matrix.csv"),
    "qwen25_7b_limit1000": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_behavior_matrix.csv"),
    "deepseek_v4_pro_limit1000": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_behavior_matrix.csv"),
}

ANALYSIS_BACKEND = "heuristic_full"
DATA_PATH = DATA_PATHS[ANALYSIS_BACKEND]
OUTPUT_DIR = Path("figures/cfa_behavior_model") / ANALYSIS_BACKEND
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# For the main heuristic full run, use all CALE variants. For strong-evaluator
# subsets, full_attack_aware_cale may be the only structured CALE variant.
CALE_VARIANTS_BY_BACKEND = {
    "heuristic_full": ["generic_cale", "attack_aware_cale", "full_attack_aware_cale"],
    "qwen25_7b_limit1000": ["full_attack_aware_cale"],
    "deepseek_v4_pro_limit1000": ["full_attack_aware_cale"],
}
CALE_VARIANTS = CALE_VARIANTS_BY_BACKEND[ANALYSIS_BACKEND]

ITEMS = [
    "misinformation_detection",
    "framing_resistance",
    "claim_status_recognition",
    "error_rejection",
    "correction_accuracy",
    "evidence_grounding",
    "source_faithfulness",
    "hallucination_control",
    "uncertainty_handling",
]

EXCLUDED_PRIMARY_CFA_COLUMNS = [
    "final_score",
    "quality_label",
    "uncertainty",
    "reference_is_nei",
    "reference_is_refutes",
    "reference_is_supports",
    "nei_uncertainty_failure_proxy",
    "refutes_correction_credit_proxy",
    "supports_status_failure_proxy",
]

candidate_paths = sorted(Path("outputs").glob("**/*behavior_matrix.csv"))
if candidate_paths:
    print("Available behavior matrix candidates:")
    for path in candidate_paths:
        print(" -", path)
else:
    print("No local behavior matrix candidates found under outputs/. If you are on the server, check the run output directory.")

print("\nSelected input:", DATA_PATH)
print("Output:", OUTPUT_DIR.resolve())

## Load And Filter CALE Rows

The primary CFA uses only CALE variants. Direct-judge rows are excluded because they do not have the CALE construct subscores and would create structural missingness.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

missing_items = [column for column in ITEMS if column not in df_raw.columns]
if missing_items:
    raise ValueError(f"Missing CFA indicator columns: {missing_items}")

df_cale = df_raw[df_raw["variant"].isin(CALE_VARIANTS)].copy()
df_items = df_cale[ITEMS].apply(pd.to_numeric, errors="coerce")

missingness = df_items.isna().mean().rename("missing_share").reset_index(names="indicator")
missingness.to_csv(OUTPUT_DIR / "cfa_indicator_missingness.csv", index=False)

df = df_items.replace([np.inf, -np.inf], np.nan).dropna().copy()

print("Raw rows:", len(df_raw))
print("CALE rows:", len(df_cale))
print("Complete CFA rows:", len(df))
display(missingness)
display(df.head())

In [ ]:
diagnostics = []
for column in ITEMS:
    values = df[column].dropna()
    unique_values = set(values.unique().tolist())
    diagnostics.append(
        {
            "indicator": column,
            "n_unique": values.nunique(),
            "min": values.min(),
            "max": values.max(),
            "mean": values.mean(),
            "std": values.std(),
            "is_binary_0_1": unique_values.issubset({0, 1}),
            "is_bounded_0_1": values.min() >= 0 and values.max() <= 1,
        }
    )

diagnostics_df = pd.DataFrame(diagnostics)
diagnostics_df.to_csv(OUTPUT_DIR / "cfa_indicator_diagnostics.csv", index=False)
display(diagnostics_df)

if diagnostics_df["is_binary_0_1"].any():
    warnings.warn(
        "Some CFA indicators are binary. semopy treats indicators as continuous, "
        "so fit indices and standard errors should be interpreted conservatively."
    )
elif diagnostics_df["is_bounded_0_1"].any():
    warnings.warn(
        "Some CFA indicators are bounded in [0, 1]. Watch for ceiling/floor effects "
        "and interpret the continuous-indicator CFA as approximate."
    )

## CFA Model Specifications

The `three_factor` model is the primary theory-guided model. The one-factor and two-factor models are comparison baselines.

In [ ]:
MODEL_SYNTAX = {
    "one_factor": """
cale_general =~ misinformation_detection + framing_resistance + claim_status_recognition + error_rejection + correction_accuracy + evidence_grounding + source_faithfulness + hallucination_control + uncertainty_handling
""",
    "two_factor": """
factual_handling =~ claim_status_recognition + correction_accuracy + evidence_grounding + source_faithfulness + hallucination_control + uncertainty_handling
adversarial_resistance =~ misinformation_detection + framing_resistance + error_rejection
factual_handling ~~ adversarial_resistance
""",
    "three_factor": """
factual_handling =~ claim_status_recognition + correction_accuracy + evidence_grounding + source_faithfulness
adversarial_resistance =~ misinformation_detection + framing_resistance + error_rejection
boundary_control =~ hallucination_control + uncertainty_handling
factual_handling ~~ adversarial_resistance
factual_handling ~~ boundary_control
adversarial_resistance ~~ boundary_control
""",
    "four_factor_sensitivity": """
claim_correction =~ claim_status_recognition + correction_accuracy
evidence_faithfulness =~ evidence_grounding + source_faithfulness
adversarial_resistance =~ misinformation_detection + framing_resistance + error_rejection
boundary_control =~ hallucination_control + uncertainty_handling
claim_correction ~~ evidence_faithfulness
claim_correction ~~ adversarial_resistance
claim_correction ~~ boundary_control
evidence_faithfulness ~~ adversarial_resistance
evidence_faithfulness ~~ boundary_control
adversarial_resistance ~~ boundary_control
""",
}

for name, syntax in MODEL_SYNTAX.items():
    print(f"\n{name}\n{'-' * len(name)}")
    print(syntax.strip())

In [ ]:
def flatten_semopy_stats(stats_obj, model_name, n_obs):
    if isinstance(stats_obj, pd.DataFrame):
        row = stats_obj.iloc[0].to_dict() if stats_obj.shape[0] == 1 else stats_obj.stack().to_dict()
    elif isinstance(stats_obj, dict):
        row = stats_obj
    else:
        row = {"stats_repr": repr(stats_obj)}

    clean = {"model": model_name, "n_obs": n_obs}
    for key, value in row.items():
        clean_key = str(key).replace(" ", "_").replace("-", "_")
        try:
            clean[clean_key] = float(value)
        except Exception:
            clean[clean_key] = value
    return clean


def fit_cfa_model(model_name, syntax, data):
    model = Model(syntax)
    fit_result = model.fit(data)
    fit_row = flatten_semopy_stats(calc_stats(model), model_name, len(data))
    fit_row["n_indicators"] = data.shape[1]
    fit_row["fit_result"] = str(fit_result)

    estimates = model.inspect(std_est=True).copy()
    estimates.insert(0, "model", model_name)
    return model, fit_row, estimates


def extract_loadings(estimates):
    observed = set(ITEMS)
    rows = estimates[(estimates["op"] == "~") & estimates["lval"].isin(observed)].copy()
    std_col = next((c for c in ["Est. Std", "Est.Std", "Std. Estimate", "Std.Estimate"] if c in rows.columns), None)
    if std_col:
        rows = rows.rename(columns={std_col: "std_loading"})

    keep = ["model", "rval", "lval", "Estimate"]
    if "std_loading" in rows.columns:
        keep.append("std_loading")
    for optional in ["Std. Err", "z-value", "p-value"]:
        if optional in rows.columns:
            keep.append(optional)

    return rows[keep].rename(columns={"rval": "factor", "lval": "indicator"}).reset_index(drop=True)

## Fit Models

In [ ]:
fitted_models = {}
fit_rows = []
estimate_tables = []

for name, syntax in MODEL_SYNTAX.items():
    print(f"Fitting {name}...")
    try:
        model, fit_row, estimates = fit_cfa_model(name, syntax, df)
        fitted_models[name] = model
        fit_rows.append(fit_row)
        estimate_tables.append(estimates)
        print(f"  done: {name}")
    except Exception as exc:
        print(f"  failed: {name}: {exc}")
        fit_rows.append({"model": name, "n_obs": len(df), "n_indicators": df.shape[1], "error": repr(exc)})

fit_indices = pd.DataFrame(fit_rows)
parameter_estimates = pd.concat(estimate_tables, ignore_index=True) if estimate_tables else pd.DataFrame()
standardized_loadings = extract_loadings(parameter_estimates) if not parameter_estimates.empty else pd.DataFrame()

fit_indices.to_csv(OUTPUT_DIR / "cfa_fit_indices.csv", index=False)
parameter_estimates.to_csv(OUTPUT_DIR / "cfa_parameter_estimates.csv", index=False)
standardized_loadings.to_csv(OUTPUT_DIR / "cfa_standardized_loadings.csv", index=False)

with (OUTPUT_DIR / "cfa_model_syntax.json").open("w", encoding="utf-8") as file:
    json.dump(MODEL_SYNTAX, file, indent=2)

display(fit_indices)
display(standardized_loadings)

In [ ]:
preferred_fit_columns = [
    "model",
    "n_obs",
    "DoF",
    "chi2",
    "chi2_p_value",
    "chi2_p-value",
    "CFI",
    "TLI",
    "RMSEA",
    "AIC",
    "BIC",
    "LogLik",
    "error",
]

comparison = fit_indices[[c for c in preferred_fit_columns if c in fit_indices.columns]].copy()
comparison.to_csv(OUTPUT_DIR / "cfa_model_comparison.csv", index=False)
display(comparison)

## Inspect Loadings / 看每个指标是否站得住

After model fit, inspect `cfa_standardized_loadings.csv` or the displayed loadings table.

Rules of thumb / 粗略判断：

- `std_loading > .70`: strong indicator，很强。
- `.40 <= std_loading <= .70`: usable/moderate，中等，可以接受。
- `.30 <= std_loading < .40`: weak but maybe interpretable，需要理论理由。
- `< .30`: weak indicator，建议 review，不要用它支撑强 claim。

If most intended indicators load moderately/strongly onto their factors:

> Standardized loadings were generally moderate to strong, suggesting that the selected diagnostic indicators contributed meaningfully to their prespecified latent dimensions.

如果某个 indicator loading 很弱，可以写：

> The weaker loading of `[indicator]` suggests that this diagnostic signal may capture behavior not fully aligned with the prespecified factor, and should be reviewed in future instrument refinement.

特别注意：如果 `boundary_control` 只有两个 indicators，它可以跑，但解释要保守。Two-indicator factors are statistically fragile and should be treated as provisional.

In [ ]:
if not standardized_loadings.empty and "std_loading" in standardized_loadings.columns:
    loading_review = standardized_loadings.copy()
    loading_review["abs_std_loading"] = loading_review["std_loading"].abs()
    loading_review["loading_flag"] = pd.cut(
        loading_review["abs_std_loading"],
        bins=[-np.inf, 0.30, 0.40, 0.70, np.inf],
        labels=["weak_review", "borderline", "moderate", "strong"],
    )
    loading_review.to_csv(OUTPUT_DIR / "cfa_loading_review.csv", index=False)
    display(loading_review.sort_values(["model", "factor", "abs_std_loading"], ascending=[True, True, False]))
else:
    print("No standardized loading column found in semopy output. Inspect cfa_parameter_estimates.csv manually.")

## Inspect Factor Correlations / 看 latent dimensions 是否可区分

Factor correlations answer whether dimensions are related but separable.

Interpretation / 解释：

- `.00-.30`: weak relation，可能是比较独立的 dimensions。
- `.30-.80`: useful range，related but distinguishable，通常最适合你的 validity story。
- `.80-.90`: very high，需要小心，可能有 strong general factor。
- `>.90`: factors may not be empirically distinguishable，不能强说是清楚分开的 latent traits。

Possible claim if correlations are moderate:

> The latent factors were positively correlated but not redundant, supporting the interpretation of CALE behavior as a set of related but distinguishable diagnostic dimensions.

Possible claim if correlations are too high:

> The high factor correlations suggest that the current indicators may be dominated by a general CALE behavior factor, and that the proposed subdimensions require further evidence or item refinement.

In [ ]:
if not parameter_estimates.empty:
    observed = set(ITEMS)
    factor_covariances = parameter_estimates[
        (parameter_estimates["op"] == "~~")
        & (~parameter_estimates["lval"].isin(observed))
        & (~parameter_estimates["rval"].isin(observed))
        & (parameter_estimates["lval"] != parameter_estimates["rval"])
    ].copy()
    std_col = next((c for c in ["Est. Std", "Est.Std", "Std. Estimate", "Std.Estimate"] if c in factor_covariances.columns), None)
    if std_col:
        factor_covariances = factor_covariances.rename(columns={std_col: "std_factor_correlation"})
    factor_covariances.to_csv(OUTPUT_DIR / "cfa_factor_correlations.csv", index=False)
    display(factor_covariances)
else:
    print("No parameter estimates available.")

## How To Interpret Model Comparison / 怎么判断模型比较

Use the table above as the first decision point.

### If `three_factor` is best or clearly better than `one_factor`

Possible claim / 可以写：

> The theory-guided three-factor CFA-style model fit the CALE behavior matrix better than a one-factor baseline, suggesting that CALE diagnostic indicators are better represented as multiple related dimensions than as a single undifferentiated quality score.

中文解释：这支持你的核心故事：CALE 的价值不是只产出一个 final score，而是把 evaluator behavior 拆成多个可诊断、可建模的 measurement dimensions。

### If `one_factor` fits similarly to `three_factor`

Possible claim / 可以写：

> The CFA-style model comparison did not clearly favor the multidimensional specification over a simpler one-factor model. This suggests that the current indicators may share a strong general CALE behavior factor, while finer-grained dimensions require further evidence.

中文解释：不能硬说三维结构被确认了。可以改讲 general CALE behavior factor 很强，subdimensions 还是 exploratory。

### If all models fit poorly

Possible claim / 可以写：

> The current CFA-style results provide limited support for the prespecified measurement model, indicating that the diagnostic indicators may need refinement or that the bounded/binary scoring procedure is not well captured by continuous-indicator CFA.

中文解释：这不是失败，反而是 measurement paper 里很合理的发现：当前 checklist indicators 可能需要修订，或者需要 ordinal CFA / WLSMV。

## Target-Specific Robustness

This repeats the model comparison for pooled, Qwen-only, and Llama-only rows. Treat these as robustness checks, not as separate fully validated models.

In [ ]:
def fit_subset(label, mask):
    subset_items = df_cale.loc[mask, ITEMS].apply(pd.to_numeric, errors="coerce")
    subset = subset_items.replace([np.inf, -np.inf], np.nan).dropna()
    rows = []
    for name, syntax in MODEL_SYNTAX.items():
        try:
            _, fit_row, _ = fit_cfa_model(name, syntax, subset)
            fit_row["subset"] = label
            rows.append(fit_row)
        except Exception as exc:
            rows.append({"subset": label, "model": name, "n_obs": len(subset), "error": repr(exc)})
    return pd.DataFrame(rows)


robustness_tables = []
all_rows = pd.Series(True, index=df_cale.index)
robustness_tables.append(fit_subset("pooled", all_rows))

if "model_name" in df_cale.columns:
    target = df_cale["model_name"].astype(str).str.lower()
    robustness_tables.append(fit_subset("qwen_only", target.str.contains("qwen", na=False)))
    robustness_tables.append(fit_subset("llama_only", target.str.contains("llama", na=False)))

if "variant" in df_cale.columns:
    for variant in CALE_VARIANTS:
        robustness_tables.append(fit_subset(f"variant_{variant}", df_cale["variant"].eq(variant)))

robustness = pd.concat(robustness_tables, ignore_index=True)
robustness.to_csv(OUTPUT_DIR / "cfa_robustness_fit_indices.csv", index=False)
display(robustness[[c for c in ["subset", *preferred_fit_columns] if c in robustness.columns]])

## Robustness Interpretation / 稳健性怎么看

Use `cfa_robustness_fit_indices.csv` to ask whether the same measurement story holds across subsets.

### If Qwen-only and Llama-only show similar model rankings

Possible claim / 可以写：

> The same model ranking was observed across target-model subsets, suggesting that the internal structure was not driven by a single target model's response distribution.

中文解释：这说明 CFA 结构不是 Qwen 或 Llama 某一个模型独有的产物。

### If variant-specific subsets show similar model rankings

Possible claim / 可以写：

> The model structure was broadly consistent across CALE variants, suggesting that the proposed dimensions reflect stable diagnostic organization rather than a variant-specific artifact.

### If robustness differs strongly across subsets

Possible claim / 可以写：

> The CFA-style structure varied across subsets, indicating that CALE's internal diagnostic organization may depend on target model, evaluator variant, or item distribution. This motivates follow-up measurement-invariance analyses.

中文解释：这时候不要说 universal structure。可以说结构可能 condition-dependent，这也很有论文价值。

## After CFA / 后续还能做什么

CFA is not the endpoint. It gives you a measurement model that can support deeper validity analyses.

### 1. Measurement invariance / 测量不变性

Question: Does the same CFA model measure the same construct across groups?

Groups to test:

- target model: Qwen vs Llama
- CALE variant: generic vs attack-aware vs full attack-aware
- framing style: neutral vs adversarial framing, if available
- dataset: FEVER vs Climate-FEVER / SciFact, if available

Claim if supported:

> The measurement structure showed preliminary invariance across target models, supporting comparisons of CALE diagnostic dimensions across model families.

### 2. Factor scores / latent dimension scores

Question: Can we use latent factor scores instead of only `final_score`?

Useful analyses:

- Which CALE variant improves `factual_handling` most?
- Which target model is weakest on `boundary_control`?
- Are NEI failures mainly boundary-control failures?
- Does adversarial framing reduce `adversarial_resistance` more than `factual_handling`?

Claim if useful:

> CFA-derived factor scores enabled dimension-specific diagnosis of evaluator behavior that would be obscured by a single aggregate score.

### 3. Criterion validity / 外部效标效度

Question: Do latent factors predict meaningful downstream failures or successes?

Possible criteria:

- `nei_uncertainty_failure_proxy`
- `supports_status_failure_proxy`
- `refutes_correction_credit_proxy`
- perturbation score shift
- label flip rate under framing
- agreement with stronger evaluator or human labels, if available

Claim if supported:

> The latent dimensions were associated with theoretically relevant downstream behavior proxies, strengthening the validity argument for CALE as a diagnostic measurement procedure.

### 4. Ordinal CFA / 更严格的 CFA

Because many indicators are binary or bounded, a stronger follow-up is ordinal CFA with WLSMV in `lavaan`.

Conservative bridge sentence:

> The present Python CFA-style analysis should be followed by ordinal CFA using robust categorical estimators before making strong claims about psychometric validation.

## Thesis-Safe Reporting Language

Use conservative wording if the notebook results go into the thesis:

> We fit theory-guided CFA-style SEM models to CALE diagnostic indicators using `semopy`, treating the bounded diagnostic scores as continuous observed variables. The primary correlated-factor model was compared with one-factor and two-factor alternatives using approximate fit indices and standardized loadings. Because several indicators are binary or bounded in `[0, 1]`, these results are interpreted as preliminary internal-structure evidence rather than definitive ordinal CFA validation.

Recommended table outputs:

- `cfa_model_comparison.csv`: model, N, df, chi-square, CFI, TLI, RMSEA, AIC, BIC.
- `cfa_standardized_loadings.csv`: factor, indicator, raw loading, standardized loading, standard error if available.
- `cfa_robustness_fit_indices.csv`: same fit summaries across target/variant subsets.